# 03 - Fetch Researchr EPC and ERC Members

In this notebook, I fetch the external committee pages from the conference
websites.

I keep `EPC` and `ERC` separate when the website has separate pages. This is
important for PLDI, because PLDI 2017-2020 has both an External Program
Committee and an External Review Committee.


## 1 - Setup

In [1]:
import re
import requests

import pandas as pd

from bs4 import BeautifulSoup
from datetime import date
from pathlib import Path
from urllib.parse import urljoin


In [2]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists() and (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError(
        "Could not find the repository root. Launch Jupyter from the repo root "
        "or set PYTHONPATH to the folder containing project_setup.py."
    )

from project_setup import setup_project

setup = setup_project()
project_folder = setup.project_folder
PROJECT = project_folder
repo = project_folder
config_path = setup.config_path
project_config = setup.project_config

run_mode = setup.run_mode
inputs_config = setup.inputs
outputs_config = setup.outputs
openalex_config = setup.openalex

allow_network = setup.allow_network
use_existing_data = setup.use_existing_data
overwrite_data = setup.overwrite_data
overwrite_artifacts = setup.overwrite_artifacts
openalex_sample_limit = setup.openalex_sample_limit
openalex_sample_include_work_ids = setup.openalex_sample_include_work_ids

step_1_data_dir = project_folder / "step_1_data"
step_1_artifacts_dir = project_folder / "step_1_artifacts"
raw_dir = step_1_data_dir / "raw"
intermediate_dir = step_1_data_dir / "intermediate"
prepared_dir = step_1_data_dir / "prepared"
summary_tables_dir = step_1_artifacts_dir / "summary_tables"
dependency_tables_dir = step_1_artifacts_dir / "dependency_tables"
check_tables_dir = step_1_artifacts_dir / "check_tables"

raw_dir.mkdir(parents=True, exist_ok=True)
intermediate_dir.mkdir(parents=True, exist_ok=True)
prepared_dir.mkdir(parents=True, exist_ok=True)
summary_tables_dir.mkdir(parents=True, exist_ok=True)
dependency_tables_dir.mkdir(parents=True, exist_ok=True)
check_tables_dir.mkdir(parents=True, exist_ok=True)

TODAY = date.today().isoformat()
RUN_FROM_CACHE = use_existing_data

print(project_folder)
print(f"Run mode: {run_mode}")


/Users/endersari/2026-02-citations-vs-pc-memberships
Run mode: fast


## 2 - External Committee URLs

In [3]:
urls_epc = {
    "ICFP": {
        2017: None, 2018: None, 2019: None, 2020: None, 2021: None,
        2022: None, 2023: None, 2024: None, 2025: None,
    },
    "POPL": {
        2017: None, 2018: None, 2019: None, 2020: None, 2021: None,
        2022: None, 2023: None, 2024: None, 2025: None,
    },
    "OOPSLA": {
        2017: "https://2017.splashcon.org/committee/splash-2017-oopsla-external-program-committee",
        2018: "https://2018.splashcon.org/committee/splash-2018-oopsla-external-program-committee",
        2019: None, 2020: None, 2021: None, 2022: None, 2023: None,
        2024: None, 2025: None,
    },
    "PLDI": {
        2017: "https://pldi17.sigplan.org/committee/pldi-2017-external-program-committee",
        2018: "https://pldi18.sigplan.org/committee/pldi-2018-external-program-committee",
        2019: "https://pldi19.sigplan.org/committee/pldi-2019-papers-external-program-committee",
        2020: "https://pldi20.sigplan.org/committee/pldi-2020-papers-external-program-committee",
        2021: None, 2022: None, 2023: None, 2024: None, 2025: None,
    },
}

urls_erc = {
    "ICFP": {
        2017: None,
        2018: "https://icfp18.sigplan.org/committee/icfp-2018-papers-external-review-committee",
        2019: "https://icfp19.sigplan.org/committee/icfp-2019-papers-external-review-committee",
        2020: "https://icfp20.sigplan.org/committee/icfp-2020-papers-external-review-committee",
        2021: None, 2022: None, 2023: None, 2024: None, 2025: None,
    },
    "POPL": {
        2017: "https://popl17.sigplan.org/committee/popl-2017-papers-external-review-committee",
        2018: None, 2019: None, 2020: None, 2021: None,
        2022: None, 2023: None, 2024: None, 2025: None,
    },
    "OOPSLA": {
        2017: None, 2018: None,
        2019: "https://2019.splashcon.org/committee/splash-2019-oopsla-external-review-committee",
        2020: "https://2020.splashcon.org/committee/splash-2020-oopsla-external-review-committee",
        2021: "https://2021.splashcon.org/committee/splash-2021-oopsla-external-review-committee",
        2022: "https://2022.splashcon.org/committee/splash-2022-oopsla-external-review---artifact-evaluation-committee",
        2023: "https://2023.splashcon.org/committee/splash-2023-oopsla-external-review---artifact-evaluation-committee",
        2024: None,
        2025: "https://2025.splashcon.org/committee/splash-2025-OOPSLA-1-external-review---artifact-evaluation-committee",
    },
    "PLDI": {
        2017: "https://pldi17.sigplan.org/committee/pldi-2017-external-review-committee",
        2018: "https://pldi18.sigplan.org/committee/pldi-2018-external-review-committee",
        2019: "https://pldi19.sigplan.org/committee/pldi-2019-papers-external-review-committee",
        2020: "https://pldi20.sigplan.org/committee/pldi-2020-papers-external-review-committee",
        2021: None, 2022: None, 2023: None, 2024: None, 2025: None,
    },
}


In [4]:
url_rows = []

for committee_type, urls in [("EPC", urls_epc), ("ERC", urls_erc)]:
    for conf, year_to_url in urls.items():
        for year, url in year_to_url.items():
            url_rows.append({
                "conference": conf,
                "year": year,
                "committee_type": committee_type,
                "url": url,
            })

url_df = pd.DataFrame(url_rows).sort_values(["committee_type", "conference", "year"])
print(url_df.shape)
display(url_df)


(72, 4)


,conference,year,committee_type,url
0,ICFP,2017,EPC,NaN
1,ICFP,2018,EPC,NaN
2,ICFP,2019,EPC,NaN
3,ICFP,2020,EPC,NaN
4,ICFP,2021,EPC,NaN
...,...,...,...,...
49,POPL,2021,ERC,NaN
50,POPL,2022,ERC,NaN
51,POPL,2023,ERC,NaN
52,POPL,2024,ERC,NaN


## 3 - Parser

In [5]:
def parse_researchr_committee(html, page_url):
    soup = BeautifulSoup(html, "html.parser")

    rows = []

    for a in soup.select('a[href*="/profile/"]'):
        h3 = a.select_one("h3.media-heading")
        if h3 is None:
            continue

        role_tags = h3.find_all("small", recursive=False)
        role_tag = role_tags[-1] if len(role_tags) else None
        if role_tag is None:
            role = "Committee Member"
        else:
            role_text = role_tag.get_text(" ", strip=True).lower()
            if "associate" in role_text and "chair" in role_text:
                role = "Associate Chair"
            elif "chair" in role_text:
                role = "Chair"
            else:
                role = "Committee Member"
            role_tag.extract()

        for marker in h3.select("sup"):
            marker.extract()

        name = h3.get_text(" ", strip=True)
        name = re.sub(r"\s+", " ", name).strip()
        if name == "":
            continue

        affiliation_tag = a.select_one("h4 span.text-black")
        if affiliation_tag is None:
            affiliation = ""
        else:
            affiliation = affiliation_tag.get_text(" ", strip=True)
            affiliation = re.sub(r"\s+", " ", affiliation)

        country_tags = a.select("h4 small")
        if len(country_tags) == 0:
            country = ""
        else:
            country = country_tags[-1].get_text(" ", strip=True)
            country = re.sub(r"\s+", " ", country)

        href = a.get("href", "")
        person_url = urljoin(page_url, href)
        researchr_id = person_url.rstrip("/").split("/profile/")[-1]

        rows.append({
            "name": name,
            "role": role,
            "affiliation": affiliation,
            "country": country,
            "person_url": person_url,
            "researchr_id": researchr_id,
        })

    return pd.DataFrame(rows)


## 4 - Fetch Pages

In [6]:
def cache_path(conf, year, committee_type):
    folder = raw_dir / conf.lower() / f"researchr_{committee_type.lower()}_htmls"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / f"{conf.lower()}{year}_{committee_type.lower()}_{TODAY}.html"


def latest_cached_page(conf, year, committee_type):
    folder = raw_dir / conf.lower() / f"researchr_{committee_type.lower()}_htmls"
    files = sorted(folder.glob(f"{conf.lower()}{year}_{committee_type.lower()}_*.html"))
    if len(files) == 0:
        return None
    return files[-1]


def get_html(conf, year, committee_type, url):
    if url is None:
        return None, "no_page", "", ""

    if RUN_FROM_CACHE:
        cached = latest_cached_page(conf, year, committee_type)
        if cached is not None:
            html = cached.read_text(encoding="utf-8", errors="replace")
            return html, "cache", str(cached), url

    response = requests.get(
        url,
        timeout=20,
        headers={"User-Agent": "Mozilla/5.0"},
    )
    response.raise_for_status()

    path = cache_path(conf, year, committee_type)
    path.write_text(response.text, encoding="utf-8")

    return response.text, "fetched", str(path), response.url


In [7]:
all_members = []
summary_rows = []

for row in url_df.itertuples(index=False):
    conf = row.conference
    year = int(row.year)
    committee_type = row.committee_type
    url = row.url if pd.notna(row.url) else None

    print(f"{committee_type} {conf} {year}...", end=" ")

    try:
        html, status, path, final_url = get_html(conf, year, committee_type, url)
        if html is None:
            df_year = pd.DataFrame()
        else:
            df_year = parse_researchr_committee(html, final_url)
        error = ""
        print(f"{status}, {len(df_year)} rows")
    except Exception as e:
        df_year = pd.DataFrame()
        status = "error"
        path = ""
        final_url = url if url is not None else ""
        error = repr(e)
        print("error")

    if len(df_year) > 0:
        df_year["conference"] = conf
        df_year["year"] = year
        df_year["committee_type"] = committee_type
        df_year["source"] = f"Researchr {committee_type}"
        df_year["source_url"] = url
        df_year["final_url"] = final_url
        all_members.append(df_year)

    summary_rows.append({
        "conference": conf,
        "year": year,
        "committee_type": committee_type,
        "url": url if url is not None else "",
        "final_url": final_url,
        "status": status,
        "cache_path": path,
        "n_members": len(df_year),
        "error": error,
    })

external_members = pd.concat(all_members, ignore_index=True)
external_summary = pd.DataFrame(summary_rows)


EPC ICFP 2017... no_page, 0 rows
EPC ICFP 2018... no_page, 0 rows
EPC ICFP 2019... no_page, 0 rows
EPC ICFP 2020... no_page, 0 rows
EPC ICFP 2021... no_page, 0 rows
EPC ICFP 2022... no_page, 0 rows
EPC ICFP 2023... no_page, 0 rows
EPC ICFP 2024... no_page, 0 rows
EPC ICFP 2025... no_page, 0 rows
EPC OOPSLA 2017... fetched, 28 rows
EPC OOPSLA 2018... fetched, 26 rows
EPC OOPSLA 2019... no_page, 0 rows
EPC OOPSLA 2020... no_page, 0 rows
EPC OOPSLA 2021... no_page, 0 rows
EPC OOPSLA 2022... no_page, 0 rows
EPC OOPSLA 2023... no_page, 0 rows
EPC OOPSLA 2024... no_page, 0 rows
EPC OOPSLA 2025... no_page, 0 rows
EPC PLDI 2017... fetched, 22 rows
EPC PLDI 2018... fetched, 20 rows
EPC PLDI 2019... fetched, 20 rows
EPC PLDI 2020... fetched, 20 rows
EPC PLDI 2021... no_page, 0 rows
EPC PLDI 2022... no_page, 0 rows
EPC PLDI 2023... no_page, 0 rows
EPC PLDI 2024... no_page, 0 rows
EPC PLDI 2025... no_page, 0 rows
EPC POPL 2017... no_page, 0 rows
EPC POPL 2018... no_page, 0 rows
EPC POPL 2019... no

## 5 - Save Data

In [8]:
external_members = external_members[
    [
        "conference", "year", "committee_type", "source", "name", "role",
        "affiliation", "country", "person_url", "researchr_id",
        "source_url", "final_url",
    ]
].sort_values(["committee_type", "conference", "year", "name"]).reset_index(drop=True)

external_summary = external_summary.sort_values(
    ["committee_type", "conference", "year"]
).reset_index(drop=True)

external_members.to_parquet(intermediate_dir / "researchr_external_members.parquet", index=False)
external_members.query("committee_type == 'EPC'").to_parquet(
    intermediate_dir / "researchr_epc_members.parquet", index=False
)
external_members.query("committee_type == 'ERC'").to_parquet(
    intermediate_dir / "researchr_erc_members.parquet", index=False
)
external_summary.to_csv(dependency_tables_dir / "researchr_external_fetch_summary.csv", index=False)

print(external_members.shape)
display(external_members.head())


(778, 12)


,conference,year,committee_type,source,name,role,affiliation,country,person_url,researchr_id,source_url,final_url
0,OOPSLA,2017,EPC,Researchr EPC,Alexander J. Summers,Committee Member,ETH Zurich,,https://2017.splashcon.org/profile/alexanderjs...,alexanderjsummers,https://2017.splashcon.org/committee/splash-20...,https://2017.splashcon.org/committee/splash-20...
1,OOPSLA,2017,EPC,Researchr EPC,Bruno C. d. S. Oliveira,Committee Member,"University of Hong Kong, China",Hong Kong,https://2017.splashcon.org/profile/brunooliveira,brunooliveira,https://2017.splashcon.org/committee/splash-20...,https://2017.splashcon.org/committee/splash-20...
2,OOPSLA,2017,EPC,Researchr EPC,Christian Hammer,Committee Member,University of Potsdam,Germany,https://2017.splashcon.org/profile/christianha...,christianhammer,https://2017.splashcon.org/committee/splash-20...,https://2017.splashcon.org/committee/splash-20...
3,OOPSLA,2017,EPC,Researchr EPC,Davide Ancona,Committee Member,University of Genova,Italy,https://2017.splashcon.org/profile/davideancona,davideancona,https://2017.splashcon.org/committee/splash-20...,https://2017.splashcon.org/committee/splash-20...
4,OOPSLA,2017,EPC,Researchr EPC,Eran Yahav,Committee Member,Technion,Israel,https://2017.splashcon.org/profile/eranyahav,eranyahav,https://2017.splashcon.org/committee/splash-20...,https://2017.splashcon.org/committee/splash-20...


## 6 - Quick Checks

In [9]:
count_table = external_summary.pivot_table(
    index=["committee_type", "year"],
    columns="conference",
    values="n_members",
    aggfunc="sum",
).fillna(0).astype(int)

display(count_table)


conference           ICFP  OOPSLA  PLDI  POPL
committee_type year                          
EPC            2017     0      28    22     0
               2018     0      26    20     0
               2019     0       0    20     0
               2020     0       0    20     0
               2021     0       0     0     0
               2022     0       0     0     0
               2023     0       0     0     0
               2024     0       0     0     0
               2025     0       0     0     0
ERC            2017     0       0    47    59
               2018    43       0    52     0
               2019    51      32    50     0
               2020    42      61    52     0
               2021     0      53     0     0
               2022     0      40     0     0
               2023     0      60     0     0
               2024     0       0     0     0
               2025     0       0     0     0

In [10]:
print("Rows by committee type")
print(external_members.groupby("committee_type").size())

print("\nRows by conference and committee type")
print(external_members.groupby(["conference", "committee_type"]).size())

print("\nFetch status")
print(external_summary["status"].value_counts())


Rows by committee type
committee_type
EPC    136
ERC    642
dtype: int64

Rows by conference and committee type
conference  committee_type
ICFP        ERC               136
OOPSLA      EPC                54
            ERC               246
PLDI        EPC                82
            ERC               201
POPL        ERC                59
dtype: int64

Fetch status
status
no_page    52
fetched    20
Name: count, dtype: int64


## 7 - What I learned

This table keeps external committee pages separate from the main PC pages.

The next step is to compare HotCRP, visible PC pages, and external committee
pages together.
